### Load Balancer

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

Intialize Multiple Inference Servers 

In [ ]:
Servers = {
    "llm_1": {
        "model": ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0
        ),
        "load": 0
    },

    "llm_2": {
        "model": ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0
        ),
        "load": 0
    },

    "llm_3": {
        "model": ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0
        ),
        "load": 0
    }
}

Nodes

In [ ]:
class State(TypedDict):
    question: str
    server: str
    answer: str

In [ ]:
def load_balancer(state: State):

    # Find server with lowest load
    server = min(
        servers,
        key=lambda x: servers[x]["load"]
    )

    # Mark server as busy
    servers[server]["load"] += 1

    print(
        f"Routing request to {server}"
    )

    return {
        "server": server
    }

Suppose the workers currently have:<br>
LLM 1 → load = 3<br>
LLM 2 → load = 1<br>
LLM 3 → load = 2

A new request comes in:<br>
LLM 1 → load = 3<br>
LLM 2 → load = 2<br>
LLM 3 → load = 2

In [ ]:
def call_llm(state: State):

    server = state["server"]

    llm = servers[server]["model"]

    response = llm.invoke(
        state["question"]
    )

    # Request finished
    servers[server]["load"] -= 1

    return {
        "answer": response.content
    }

Simple Graph

In [ ]:
graph = StateGraph(State)

graph.add_node(
    "load_balancer",
    load_balancer
)

graph.add_node(
    "call_llm",
    call_llm
)

graph.set_entry_point(
    "load_balancer"
)

graph.add_edge(
    "load_balancer",
    "call_llm"
)

graph.add_edge(
    "call_llm",
    END
)

In [ ]:
app = graph.compile()

#### RUN

In [ ]:
while True:

    question = input("\nAsk: ")

    if question.lower() == "exit":
        break

    result = app.invoke({
        "question": question,
        "worker": "",
        "answer": ""
    })

    print("\nAnswer:", result["answer"])